## Configuration

In [ ]:
DATA_PATH             = "../data/cleaned_loan_data.csv"
LABEL_COLUMN          = "loan_approved"
TASK_TYPE             = "binary"
TOP_N                 = 3
OUTPUT_DIR            = "models"

USERNAME              = "your-username"
CLUSTER_DOMAIN        = "apps.cluster.example.com"
REGISTERED_MODEL_NAME = "loan-default-predictor"
VERSION               = "0.0.1"

## 1. Load & split data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=[LABEL_COLUMN])
y = df[LABEL_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

train_df = pd.concat([X_train, y_train], axis=1)
test_df  = pd.concat([X_test,  y_test],  axis=1)

print(f"Train: {len(train_df)} rows  |  Test: {len(test_df)} rows")

## 2. Train — model selection stage

In [ ]:
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(
    problem_type=TASK_TYPE,
    label=LABEL_COLUMN,
    eval_metric="accuracy",
    path=f"{OUTPUT_DIR}/predictor",
    verbosity=2,
).fit(
    train_data=train_df,
    num_stack_levels=3,
    num_bag_folds=2,
    use_bag_holdout=True,
    holdout_frac=0.2,
    time_limit=3600,
    presets="medium_quality",
)

## 3. Leaderboard — pick top N models

In [ ]:
leaderboard = predictor.leaderboard(test_df)
top_models = leaderboard.head(TOP_N)["model"].tolist()
print("Top models:", top_models)
leaderboard.head(TOP_N)

## 4. Refit top models on full data

In [ ]:
import os
import json
from pathlib import Path

for model_name in top_models:
    print(f"\nRefitting {model_name}...")
    model_name_full = model_name + "_FULL"
    output_path = Path(OUTPUT_DIR) / model_name_full

    clone = predictor.clone(
        path=output_path / "predictor",
        return_clone=True,
        dirs_exist_ok=True,
    )
    clone.delete_models(models_to_keep=[model_name])
    clone.refit_full(model=model_name)
    clone.set_model_best(model=model_name_full, save_trainer=True)
    clone.save_space()

    eval_results = clone.evaluate(test_df)
    os.makedirs(output_path / "metrics", exist_ok=True)
    with open(output_path / "metrics" / "metrics.json", "w") as f:
        json.dump(eval_results, f, indent=2)

    print(f"  Saved to {output_path}")
    print(f"  Metrics: {eval_results}")

## 5. Summary

In [ ]:
print("Refitted models saved under:", OUTPUT_DIR)
for model_name in top_models:
    path = Path(OUTPUT_DIR) / (model_name + "_FULL")
    metrics_file = path / "metrics" / "metrics.json"
    with open(metrics_file) as f:
        metrics = json.load(f)
    print(f"  {model_name}_FULL: {metrics}")

## 6. Push best model to OCI registry

In [ ]:
# Find best model by accuracy
best_model = max(
    top_models,
    key=lambda m: json.load(open(Path(OUTPUT_DIR) / (m + "_FULL") / "metrics" / "metrics.json"))["accuracy"],
)
print(f"Best model: {best_model}_FULL")

deploy_path = Path("best_model")
best_predictor = TabularPredictor.load(str(Path(OUTPUT_DIR) / (best_model + "_FULL") / "predictor"))
best_predictor.clone_for_deployment(path=str(deploy_path))

In [ ]:
%%bash
oc registry login \
  --to=/tmp/ocp-auth.json

In [ ]:
%env REGISTRY_AUTH_FILE=/tmp/ocp-auth.json

In [ ]:
import os
from model_registry.utils import save_to_oci_registry

oci_ref = f"default-route-openshift-image-registry.{CLUSTER_DOMAIN}/{USERNAME}/{REGISTERED_MODEL_NAME}:{VERSION}"
save_to_oci_registry(
    base_image="registry.access.redhat.com/ubi9/ubi-minimal:9.4",
    oci_ref=oci_ref,
    model_files_path=str(deploy_path),
)
print(f"Pushed to {oci_ref}")